In [2]:
import numpy as np
import pandas as pd

In [3]:
df_Amoxycillin = pd.read_csv("dataset_amoxy.csv", low_memory=False)

In [4]:
df_Amoxycillin.reset_index(inplace= True)
df_Amoxycillin.columns

Index(['index', 'Unnamed: 0', 'Species', 'Organism Group', 'Country', 'Year',
       'Source', 'Value', 'Value_I'],
      dtype='object')

In [5]:
df_Amoxycillin

,index,Unnamed: 0,Species,Organism Group,Country,Year,Source,Value,Value_I
0,0,0,Serratia marcescens,Enterobacteriaceae,France,2013,CVS,32.00,Resistant
1,1,1,Serratia marcescens,Enterobacteriaceae,France,2013,RSP,32.00,Resistant
2,2,2,Enterobacter cloacae,Enterobacteriaceae,France,2013,BF,32.00,Resistant
3,3,3,Escherichia coli,Enterobacteriaceae,France,2013,GU,4.00,Susceptible
4,4,4,Haemophilus influenzae,Haemophilus spp,France,2013,RSP,0.25,Susceptible
...,...,...,...,...,...,...,...,...,...
323651,323651,323651,Klebsiella pneumoniae,Enterobacteriaceae,France,2013,RSP,16.00,Intermediate
323652,323652,323652,Escherichia coli,Enterobacteriaceae,France,2013,GU,2.00,Susceptible
323653,323653,323653,Escherichia coli,Enterobacteriaceae,France,2013,GU,8.00,Susceptible
323654,323654,323654,Escherichia coli,Enterobacteriaceae,France,2013,GU,8.00,Susceptible


In [6]:
df = df_Amoxycillin[df_Amoxycillin.Value.notna()]
df = df.reset_index()

In [7]:
df.columns

Index(['level_0', 'index', 'Unnamed: 0', 'Species', 'Organism Group',
       'Country', 'Year', 'Source', 'Value', 'Value_I'],
      dtype='object')

In [8]:
df = df[df.columns.drop(['level_0', 'index', 'Unnamed: 0'])]

In [9]:
df = df[df.Value_I.notna()]

In [10]:
df.index = [i for i in range(323656)]

In [11]:
df

,Species,Organism Group,Country,Year,Source,Value,Value_I
0,Serratia marcescens,Enterobacteriaceae,France,2013,CVS,32.00,Resistant
1,Serratia marcescens,Enterobacteriaceae,France,2013,RSP,32.00,Resistant
2,Enterobacter cloacae,Enterobacteriaceae,France,2013,BF,32.00,Resistant
3,Escherichia coli,Enterobacteriaceae,France,2013,GU,4.00,Susceptible
4,Haemophilus influenzae,Haemophilus spp,France,2013,RSP,0.25,Susceptible
...,...,...,...,...,...,...,...
323651,Klebsiella pneumoniae,Enterobacteriaceae,France,2013,RSP,16.00,Intermediate
323652,Escherichia coli,Enterobacteriaceae,France,2013,GU,2.00,Susceptible
323653,Escherichia coli,Enterobacteriaceae,France,2013,GU,8.00,Susceptible
323654,Escherichia coli,Enterobacteriaceae,France,2013,GU,8.00,Susceptible


In [12]:
df['Year_c'] = df['Year'] - df['Year'].mean()
species_idx, species = pd.factorize(df["Species"])
country_idx, country = pd.factorize(df["Country"])
source_idx, source = pd.factorize(df["Source"])

df["species_idx"] = species_idx
df["country_idx"] = country_idx
df["source_idx"] = source_idx

In [13]:
df

,Species,Organism Group,Country,Year,Source,Value,Value_I,Year_c,species_idx,country_idx,source_idx
0,Serratia marcescens,Enterobacteriaceae,France,2013,CVS,32.00,Resistant,1.306446,0,0,0
1,Serratia marcescens,Enterobacteriaceae,France,2013,RSP,32.00,Resistant,1.306446,0,0,1
2,Enterobacter cloacae,Enterobacteriaceae,France,2013,BF,32.00,Resistant,1.306446,1,0,2
3,Escherichia coli,Enterobacteriaceae,France,2013,GU,4.00,Susceptible,1.306446,2,0,3
4,Haemophilus influenzae,Haemophilus spp,France,2013,RSP,0.25,Susceptible,1.306446,3,0,1
...,...,...,...,...,...,...,...,...,...,...,...
323651,Klebsiella pneumoniae,Enterobacteriaceae,France,2013,RSP,16.00,Intermediate,1.306446,4,0,1
323652,Escherichia coli,Enterobacteriaceae,France,2013,GU,2.00,Susceptible,1.306446,2,0,3
323653,Escherichia coli,Enterobacteriaceae,France,2013,GU,8.00,Susceptible,1.306446,2,0,3
323654,Escherichia coli,Enterobacteriaceae,France,2013,GU,8.00,Susceptible,1.306446,2,0,3


In [18]:
l = []
for i in df.Value_I:
    if(i == 'Resistant'):
        l.append(1)
    else:
        l.append(0)

df['resistant'] = l
del(l)

In [19]:
n_species = len(species)
n_country = len(country)
n_source = len(source)

In [ ]:
import pymc as pm

with pm.Model() as model:

    intercept = pm.Normal("intercept", 0, 5)
    beta_year = pm.Normal("beta_year", 0, 1)
    
    beta_source = pm.Normal("beta_source", 0, 1, shape=n_source)

    sigma_species = pm.HalfNormal("sigma_species", 1)
    sigma_country = pm.HalfNormal("sigma_country", 1)

    species_effect = pm.Normal("species_effect", 0, sigma_species, shape=n_species)
    country_effect = pm.Normal("country_effect", 0, sigma_country, shape=n_country)

    # linear predictor
    logit_p = (
        intercept
        + beta_year * df["Year_c"].values
        + beta_source[df["source_idx"].values]
        + species_effect[df["species_idx"].values]
        + country_effect[df["country_idx"].values]
    )

    # likelihood
    y = pm.Bernoulli(
        "y",
        logit_p=logit_p,
        observed=df["resistant"].values)

/home/dark/Desktop/Project2/.venv/lib/python3.12/site-packages/arviz/__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


In [21]:
with model:
    trace = pm.sample(
        1000,
        tune=1000,
        chains=4,
        target_accept=0.9
    )

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [intercept, beta_year, beta_source, sigma_species, sigma_country, species_effect, country_effect]


/home/dark/Desktop/Project2/.venv/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" 
for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

ValueError: Not enough samples to build a trace.